# Episode 5: LiveAgent — Real-Time Voice

So far every agent has been the `Agent` class: you `ask()`, you get a reply. This episode shows off something different — the **`LiveAgent`**.

**In this episode you'll build:** a voice assistant you can actually talk to.

> **A showcase, on purpose.** `LiveAgent` is a *different class* from `Agent`. It does **not** have `ask()`. Instead it opens a continuous, full-duplex audio session — you and the agent can speak at the same time. We put it here, early, so you see the breadth of AG2 Beta before going deep.

## How `LiveAgent` differs from `Agent`

| | `Agent` | `LiveAgent` |
|---|---|---|
| Interaction | request / response | continuous audio session |
| Call style | `reply = await agent.ask(...)` | `async with agent.run() as context:` |
| Config | `OpenAIConfig` | `openai.RealTimeConfig` |
| Input / output | text | microphone audio / speaker audio |

A `LiveAgent` holds a **realtime** config and is opened with `agent.run()`, which yields a `ConversationContext`. Audio peers — a recorder and a player — share that context so audio flows in and out continuously, with built-in voice activity detection and barge-in.

## Step 1: Install the audio dependencies

The voice session needs microphone and speaker access via `sounddevice`.

In [ ]:
!pip install sounddevice numpy -q

## Step 2: Build a LiveAgent

The constructor takes a `name`, a `prompt`, and a realtime `config`. `openai.RealTimeConfig` picks the realtime model and — via `AudioOutput` — the voice.

Creating the agent makes no network or audio connection, so this cell is safe to run anywhere.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen.beta.live import LiveAgent, openai

voice_bot = LiveAgent(
    name="voice_bot",
    prompt="You are a friendly voice assistant. Keep responses brief.",
    config=openai.RealTimeConfig(
        "gpt-realtime-2",
        output=openai.AudioOutput(voice="ballad"),
    ),
)
print(type(voice_bot).__name__, "ready:", voice_bot.name)

## Step 3: Run the voice session

Opening the session uses `agent.run()` as an async context manager. A `SoundDevicePlayer` and `SoundDeviceRecorder` share the same `context` so the recorder's audio reaches the model and the model's audio reaches your speakers.

> **⚠️ Run this as a script, not in the notebook.** It opens your microphone and runs until you stop it with `Ctrl+C` — an infinite loop, so it would hang a notebook cell. Save it as `live_voice.py` and run `python live_voice.py` in a terminal.

```python
# live_voice.py  --  save and run from a terminal:  python live_voice.py
import asyncio

from autogen.beta.live import (
    LiveAgent,
    SoundDevicePlayer,
    SoundDeviceRecorder,
    openai,
)

agent = LiveAgent(
    name="voice_bot",
    prompt="You are a friendly voice assistant. Keep responses brief.",
    config=openai.RealTimeConfig(
        "gpt-realtime-2",
        output=openai.AudioOutput(voice="ballad"),
    ),
)


async def main() -> None:
    async with (
        agent.run() as context,            # yields a ConversationContext
        SoundDevicePlayer(context=context),     # speaker out
        SoundDeviceRecorder(context=context),   # microphone in
    ):
        print("Listening... press Ctrl+C to stop.")
        await asyncio.Future()  # run until interrupted


if __name__ == "__main__":
    asyncio.run(main())
```

## Watching the transcript

The realtime provider streams a text transcript alongside the audio. Subscribe to `ModelMessageChunk` on the context's stream to print the assistant's words as they arrive.

This is also a script — same reason as above.

```python
# live_transcript.py  --  save and run from a terminal
import asyncio

from autogen.beta.events import ModelMessageChunk
from autogen.beta.live import (
    LiveAgent,
    SoundDevicePlayer,
    SoundDeviceRecorder,
    openai,
)

agent = LiveAgent(
    name="voice_bot",
    prompt="You are a friendly voice assistant. Keep responses brief.",
    config=openai.RealTimeConfig(
        "gpt-realtime-2",
        output=openai.AudioOutput(voice="ballad"),
    ),
)


async def main() -> None:
    async with (
        agent.run() as context,
        SoundDevicePlayer(context=context),
        SoundDeviceRecorder(context=context),
    ):
        print("Listening... press Ctrl+C to stop.")
        with context.stream.where(ModelMessageChunk).join() as events:
            async for event in events:
                print(event)


if __name__ == "__main__":
    asyncio.run(main())
```

## Additional content

**Text-only output.** Swap `openai.AudioOutput(...)` for `openai.TextOutput()` to keep the low-latency realtime session but receive text instead of synthesized audio.

**Tools in a live session.** `LiveAgent` supports the same `@agent.tool` decorator as a regular `Agent`. Tool calls are routed through AG2's normal executor and results are sent back into the realtime session automatically.

**Why a separate class?** Realtime voice is a fundamentally different interaction model — streaming, bidirectional, interruptible. Rather than bolt that onto `Agent`, AG2 Beta gives it the purpose-built `LiveAgent`.

## What's next

That wraps up Group 1 — the fundamentals. Next, Group 2 opens up the **Agent harness**: middleware, assembly policies, and structured output — the slots that turn a basic agent into a production-ready one.